In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from einops import rearrange

In [2]:
# Configuration flags and hyperparameters
USE_MAMBA = 1
DIFFERENT_H_STATES_RECURRENT_UPDATE_MECHANISM = 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [8]:
d_model = 8
state_size = 128  # Example state size
seq_len = 100  # Example sequence length
batch_size = 256  # Example batch size
last_batch_size = 81  # only for the very last batch of the dataset
current_batch_size = batch_size
different_batch_size = False
h_new = None
temp_buffer = None

In [4]:
class S6(nn.Module):
    def __init__(self,seq_len, d_model, state_size,device):
        super(S6, self).__init__()

        self.fc1 = nn.Linear(d_model, d_model, device=device)
        self.fc2 = nn.Linear(d_model, state_size, device=device)
        self.fc3 = nn.Linear(d_model, state_size, device=device)

        self.seq_len = seq_len
        self.d_model = d_model
        self.state_size = state_size

        self.A = nn.Parameter(F.normalize(torch.ones(d_model, state_size, device=device), p=2, dim=-1))
        nn.init.xavier_uniform_(self.A)

        self.B = torch.zeros(batch_size, self.seq_len, self.state_size, device=device)
        self.C = torch.zeros(batch_size, self.seq_len, self.state_size, device=device)

        self.delta = torch.zeros(batch_size, self.seq_len, self.d_model, device=device)
        self.dA = torch.zeros(batch_size, self.seq_len, self.d_model, self.state_size, device=device)
        self.dB = torch.zeros(batch_size, self.seq_len, self.d_model, self.state_size, device=device)

        # h [batch_size, seq_len, d_model, state_size]
        self.h = torch.zeros(batch_size, self.seq_len, self.d_model, self.state_size, device=device)
        self.y = torch.zeros(batch_size, self.seq_len, self.d_model, device=device)

    def discretization(self):

        self.dB = torch.einsum("bld, bln -> bldn", self.delta, self.B)
        self.dA = torch.einsum("bld, bln -> bldn", self.delta, self.A)

        return self.dA, self.dB
    
    def forward(self, x):
        # 参考 Mamba 论文 伪码
        self.B = self.fc2(x)
        self.C = self.fc3(x)
        self.delta = F.softplus(self.fc1(x))

        # 离散化
        self.discretization()

        if DIFFERENT_H_STATES_RECURRENT_UPDATE_MECHANISM:
            global current_batch_size
            current_batch_size = x.shape[0]

            if self.h.shape[0] != current_batch_size:
                different_batch_size = True

                h_new = torch.einsum("bldn, bldn -> bldn", self.dA, self.h[:current_batch_size,...]) + rearrange(x, "b l d -> b l d 1") * self.dB

            else:
                different_batch_size = False
                h_new = torch.einsum("bldn, bldn -> bldn", self.dA, self.h) + rearrange(x, "b l d -> b l d 1") * self.dB
            
            # y  [batch_size, seq_len, d_model]
            self.y = torch.einsum("bln, bldn -> bld", self.C, h_new)

            global temp_buffer
            temp_buffer = h_new.detach().clone() if not self.h.requires_grad else h_new.clone()

            return self.y
        
        else:
            # h [batch_size, seq_len, d_model, state_size]
            h = torch.zeros(x.size(0), self.seq_len, self.d_model, self.state_size, device=x.device)
            y = torch.zeros_like(x)

            h = torch.einsum("bldn, bldn -> bldn",self.dA, h) + rearrange(x, "b l d -> b l d 1") * self.dB

            # y  [batch_size, seq_len, d_model]
            y = torch.einsum("bln, bldn -> bld",self.C, h)

            return y

In [5]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device: str = 'cuda'):
        super(RMSNorm,self).__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model, device=device))

    def forward(self, x):
        output = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight
        
        return output

In [6]:
class MambaBlock(nn.Module):
    def __init__(self, seq_len, d_model, state_size, device):
        super(MambaBlock,self).__init__()

        self.inp_proj = nn.Linear(d_model, 2*d_model, device=device)
        self.out_proj = nn.Linear(2*d_model, d_model, device=device)

        # 残差连接: y = D *x
        self.D = nn.Linear(d_model, 2*d_model, device=device)

        # 在偏置项设置属性 _no_weight_decay
        self.out_proj.bias._no_weight_decay = True
        
        # 将偏置项 初始化一个 小的 常数值
        nn.init.constant_(self.out_proj.bias, 1.0)

        self.S6 = S6(seq_len, 2*d_model, state_size, device)

        # 添加一个使用3x3卷积核的1D卷积层
        self.conv = nn.Conv1d(seq_len, seq_len, kernel_size=3, padding=1, device=device)
        
        # 在卷积输出上添加线性层
        self.conv_linear = nn.Linear(2*d_model, 2*d_model, device=device)

        # 归一化层
        self.norm = RMSNorm(d_model, device=device)

    def forward(self, x):

        # 参考MAMBA论文中的图3
        x = self.norm(x)

        x_proj = self.inp_proj(x)

        # 添加一个使用3x3卷积核的1D卷积层
        x_conv = self.conv(x_proj)

        x_conv_act = F.silu(x_conv)

        # 在卷积输出上添加线性层
        x_conv_out = self.conv_linear(x_conv_act)

        x_ssm = self.S6(x_conv_out)
        x_act = F.silu(x_ssm) #  激活函数的作用: x * sigmoid(x) 

        # 通过乘法引入非线性的残差跳过连接
        x_residual = F.silu(self.D(x))

        x_combined = x_act * x_residual

        x_out = self.out_proj(x_combined)

        return x_out

In [7]:
class Mamba(nn.Module):
    def __init__(self, seq_len, d_model, state_size, device):
        super(Mamba, self).__init__()
        self.mamba_block1 = MambaBlock(seq_len, d_model, state_size, device)
        self.mamba_block2 = MambaBlock(seq_len, d_model, state_size, device)
        self.mamba_block3 = MambaBlock(seq_len, d_model, state_size, device)

    def forward(self,x):
        x = self.mamba_block1(x)
        x = self.mamba_block2(x)
        x = self.mamba_block3(x)
        return x

In [12]:
x = torch.rand(batch_size, seq_len, d_model, device=device) # [256, 100, 8]
mamba = Mamba(seq_len, d_model, state_size, device)

norm = RMSNorm(d_model)
x = norm(x)

test_output = mamba(x)
print(f"test_output.shape = {test_output.shape}")

RuntimeError: einsum(): the number of subscripts in the equation (3) does not match the number of dimensions (2) for operand 1 and no ellipsis was given